# Customer Support Multi-Agent System using LangGraph

### Install Dependencies

In [ ]:
!pip install -U \
    langchain==0.3.27 \
    langgraph==0.6.4 \
    langchain-openai==0.3.28 \
    langchain-tavily==0.2.11 \
    rich==14.1.0 \
    pandas==2.3.2 \
    matplotlib==3.10.5
    python-dotenv

### Set up API keys

In [ ]:
import os

# OpenAI
os.environ['AZURE_OPENAI_API_KEY'] = '' # Add your api key here
os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://lsjjbtesjrrep.openai.azure.com/'
os.environ['OPENAI_API_VERSION'] = '2023-05-15'


#Tavily
os.environ["TAVILY_API_KEY"] = "" # Add your api key here

### Import Necessary Modules

In [ ]:
from langchain.agents.agent_types import AgentType 
from langchain_openai import AzureChatOpenAI
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import AIMessage
from langgraph.prebuilt import create_react_agent, InjectedState
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
import pandas as pd
import matplotlib.pyplot as plt
from rich import print 


### Create a Search Tool

In [ ]:
from langchain_tavily import TavilySearch
web_search_tool = TavilySearch(max_results=1)

### Initialise and LLM

In [ ]:
model = AzureChatOpenAI(
    azure_deployment= "gpt-4.1-mini",
    model="gpt-4.1-mini",
    temperature=0.7
)

### Create a handoff tool generator

In [ ]:
def create_handoff_tool(*, agent_name: str, description: str | None = None):
    name = f"transfer_to_{agent_name}"  # add instructions to transfer to agent
    description = description or f'Transfer to {agent_name}'  # add description or instructions to transfer to agent

    @tool(name, description=description)
    def handoff_tool(state: Annotated[dict, InjectedState],
                     tool_call_id:Annotated[str, InjectedToolCallId]):
        msg = ToolMessage(content=f"Transferred to {agent_name}", tool_call_id=tool_call_id)
        return Command(goto=agent_name, graph=Command.PARENT,
                       update={"messages": state["messages"] + [msg]}
                      )
        
     # define handoff_tool in the same way it was defined in the previous module activity

    return handoff_tool

### Define handoff tools

In [ ]:
assign_to_refund_agent = create_handoff_tool(agent_name="RefundAgent") # Tool to route refund related queries to the RefundAgent

assign_to_sales_agent = create_handoff_tool(agent_name="SalesAgent") # Tool to route sales queries to the SalesAgent

### Create a supervisor agent


In [ ]:
supervisor_agent = create_react_agent(
    model=model,
    tools=[assign_to_refund_agent, assign_to_sales_agent],
    prompt= ''' 
    You are a supervisor agent. Your job is to supervise the other two agents and coordinate activities. 
    For any refund requests hand over to the RefundAgent.
    For any sales requests hand over to SalesAgent
    
    ''',
    name= "SupervisorAgent",

) # create supervision agent

### Read mock refund status dataset

In [ ]:
refund_data = pd.read_csv("refunds_status.csv") # read csv file

### Perform basic EDA

In [ ]:
# 1. Check the structure of the dataframe
print("Dataframe Info:")
print(refund_data.info())
print("\n")

# 2. Preview the first few rows
print("First 5 rows of the dataframe:")
print(refund_data.head())
print("\n")

# 3. Check for missing values
print("Missing values per column:")
print(refund_data.isnull().sum())
print("\n")

# 4. Check for duplicate order_ids
duplicate_count = refund_data["order_id"].duplicated().sum()
print(f"Number of duplicate order_ids: {duplicate_count}")
print("\n")

# 5. Plot the distribution of refund statuses
print("Status distribution:")
print(refund_data["status"].value_counts())

refund_data["status"].value_counts().plot(kind="bar", title="Refund Status Distribution", color="skyblue")
plt.xlabel("Status")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

Create a `check_refund_status` tool. This tool looks up a customer's refund status given order_id and searches the DataFrame for matching orders. If fount, it returns the sthatus of the order from that row, otherwiese it returns "Not Found"

In [ ]:
def check_refund_status(order_id: int) -> str:
    """Check the status of a customer refund based on the order id.
    """
    match = refund_data.loc[refund_data["order_id"] == order_id]
    if not match.empty:
        return match.iloc[0]["status"]
    return "Not found"


### Create a Refund agent


In [ ]:
refund_agent = create_react_agent(
    model=model,
    tools=[check_refund_status],
    prompt="You are a refund agent, a helpful assistant who checks the order status of customers. ",
    name="refund_agent",
)

### Create a Sales Agent

In [ ]:
sales_agent = create_react_agent(
    model=model,
    tools=[web_search_tool],
    prompt="You are a sales agent. Help users with product details, availability, and general sales support by searching online. Always search on this website: https://azqueenbee.com/collections/beeswax-products?srsltid=AfmBOop2N7DstyWJp4c-tiZ-BuWQh89aAuJPAUO10BloE0HtmFZBGeBn",
    name="sales_agent",
)

## Compile Supervisor Graph using LangGraph

In [ ]:
supervisor_graph = (
    StateGraph(MessagesState)
    .add_node(supervisor_agent, destinations=("refund_agent", "sales_agent", END))
    .add_node(refund_agent)
    .add_node(sales_agent)
    .add_edge(START, "supervisor")
    .compile()
)